# AICIDENTIA - 01 Data audit

**Question:** What is in the AIID snapshot, and is it fit for purpose? (Sheets, columns, missingness, duplicates, date coverage, labels, entities, reports, snapshot.)

> CRISP-DM phase: **Data Understanding**

All logic lives in `aicidentia.data.audit`; this notebook runs it, shows the results and explains them. Numbers shown here are read from `reports/results.json` (keys `audit.*`) or from tables, never typed by hand. The full write-up, with a DECISIONS section, is `docs/data_audit.md`.

In [ ]:
from IPython.display import Image, Markdown, display

from aicidentia import registry
from aicidentia.config import PROJECT_ROOT, configure_logging, load_config, set_seeds

configure_logging()
cfg = load_config()
SEED = set_seeds(cfg["random_seed"])
FIG = PROJECT_ROOT / cfg["paths"]["figures"]
print(f"{cfg['project_name']} | seed {SEED} | snapshot {cfg['snapshot_date']}")

## 1. Run the audit

`run_audit()` reads the Excel export and the snapshot from `data/raw/` (never modified) and returns tables and scalar facts. `write_all()` records them in the results registry and writes tables, figures and the two documents.

In [ ]:
from aicidentia.data.audit import run_audit
from aicidentia.data.audit_outputs import write_all

res = run_audit()
write_all(res)
t, s = res.tables, res.scalars

## 2. Files, sheets and the Data Dictionary

The sheets carry banner rows above the header, and the row counts in the banners are an independent check on loading. The Data Dictionary states fill rates; the table below shows only columns where the observed rate disagrees by more than one percentage point.

In [ ]:
display(t["sheet_shapes"])
dd = t["dictionary_crosscheck"]
display(dd[dd["flag"] != ""][["column", "group", "stated_fill", "observed_fill", "flag"]])

## 3. Incidents over time

Dates are when the harm occurred. The snapshot year is partial and is excluded from trend tests; counts grow steeply, so shares (not counts) will be the primary trend measure.

In [ ]:
display(t["incidents_per_year"].query("year >= 2010"))
display(Image(filename=FIG / "fig_audit_1_incidents_per_year.png"))

## 4. Duplicates and text length

In [ ]:
print({k: v for k, v in s["duplicates"].items() if k != "merge_chains"})
print("merge chains:", s["duplicates"]["merge_chains"])
display(t["near_duplicate_pairs_top25"].head(5))
display(t["text_length_stats"].round(1))
display(Image(filename=FIG / "fig_audit_4_text_lengths.png"))

## 5. Labels: coverage, classes, balance

The MIT taxonomy is the only full-coverage risk-domain label, and the snapshot's taxonomy record marks it as **automatically generated**. Human-assigned CSETv1 and GMF cover far fewer incidents. MIT coverage also falls for recent years, which matters for any time-based split.

In [ ]:
print("Taxonomy provenance:", s["taxonomy_provenance"])
display(t["label_profile"].query("group == 'MIT'").round(3))
display(t["mit_domain_distribution"])
display(t["mit_domain_by_period"])
display(Image(filename=FIG / "fig_audit_2_taxonomy_coverage.png"))
display(Image(filename=FIG / "fig_audit_3_mit_domain_balance.png"))

## 6. Entities

Excel stores entities as comma-joined, title-cased names (lossy, and ambiguous when a name contains a comma); the snapshot stores exact slugs that join to the Entities sheet. The top lists show that these fields mix organisations with generic actors, technologies and abstract values.

In [ ]:
display(t["entity_roles_summary"].drop(columns=["excel_example"]).round(2))
for role in ["deployer", "harmed"]:
    display(t[f"entities_top20_{role}"].head(10))

## 7. Reports and snapshot

In [ ]:
rep = s["reports"]
print(
    {
        k: rep[k]
        for k in [
            "reports_sheet_rows",
            "stub_reports",
            "reports_not_linked_to_any_incident",
            "incidents_with_one_report",
            "report_count_median",
            "incidents_with_any_full_text_pct",
            "report_text_words_median",
        ]
    }
)
print("Editor-marked similarity:", s["similarity"])
display(Image(filename=FIG / "fig_audit_5_reports_per_incident.png"))

## 8. Registry check

The same facts are in the results registry, with the file and function that produced each one.

In [ ]:
for key in [
    "audit.n_incidents",
    "audit.mit_domain_n",
    "audit.similarity.editor_similar_usable_queries_symmetrised",
    "audit.year_range_recommendation",
]:
    print(key, "=", registry.get(key))

## 9. Decisions

Rendered from `docs/data_audit.md` so the notebook and the document cannot disagree.

In [ ]:
text = (PROJECT_ROOT / "docs" / "data_audit.md").read_text(encoding="utf-8")
display(Markdown(text[text.index("## DECISIONS") :]))